## Issues
1. Capacity availability on certain date
   1. #of trucks available
2. Look at the transmode
   1. (Truck - Container) 40Ft truck  
3. Convert (shipment_plans) Units into pallets
4. Container should be limited by weight, area utilization, Volume
   1. Add-on: Conditions are Configurable from the user
   2. Calculate golden ratio (Currently A -> B)
5. Pull-in method


# Code

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import os

from objects.ContainerFleetOptimizationResult import ContainerFleetOptimizationResult
from objects.ContainerOptimizationResult import ContainerOptimizationResult
from pandas import DataFrame, merge, concat

from numpy import floor, ceil, cumsum, where
from collections import defaultdict
from logging import getLogger
import math, time

## Classes

from objects.Axle import Axle
from objects.Container import Container
from objects.ContainerSummary import ContainerSummary
from objects.ContainerLoadingRules import ContainerLoadingRules
from objects.Dimension import Dimension
from objects.Pallet import Pallet
from objects.Position import Position
import uuid

from objects.AxleLoadResult import AxleLoadResult
from objects.UtilizationMetrics import UtilizationMetrics
from objects.RemainingCapacity import RemainingCapacity


from __future__ import annotations
import math
import hashlib
from datetime import datetime
from typing import List, Optional, Tuple, Dict, Any
import pandas as pd
from objects.ShipmentGroup import ShipmentGroup
import json

##
from utils.visualize import container_visualization
from utils.measure_conversion import *

logger = getLogger("load_planner")


# Assumptions
1. Units in the pallet are homogeneous
2. Units are calculated from item quantity to pallets
3. Dimensions are measured in inches (Later converted to Feet / other metrics)
4. **Routes have been pre-planned**
5. 

# Solver
1. Decide what items to load based on:
- Delivery date, priority, Item name, 
- Convert units into pallets
- Confirm the #of units shipped & update 
2. Grouping logic:
- Combine all items

### Optimizers to look into
1. 
## Feature enhancements
1. Stock pull-in from future (Early shipping)
2. Axle based handling-unit 📦(container) positioning
3. 

In [3]:
### Writing results to Database ###
# Write results to tables:
# 1. Handling_unit
# 2. Handling_unit_content
# 3. Handling_unit_position
# 4. Transport_equipment_assignment
# 5. route_planned


In [4]:
### Verify loaded 🚚 truck_equipment_assignment & handling_unit positions inside the container  

## Establish connection with Neon Database

In [5]:
### NeonDB Connection
import database.helper as db_helper
db_conn = db_helper.create_connection()

## Loading Data
item_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.item_master", connection=db_conn)
lane_master_df = db_helper.fetch_data(sql="select * from inventory_management.public.lane_master", connection=db_conn)
load_equipment_metadata_df = db_helper.fetch_data(sql="select * from inventory_management.public.load_equipment_metadata", connection=db_conn)
location_df = db_helper.fetch_data(sql="select * from inventory_management.public.location", connection=db_conn)
shipment_demand_df = db_helper.fetch_data(sql="select * from inventory_management.public.shipment_plans", connection=db_conn)
sku_uom_df = db_helper.fetch_data(sql="select * from inventory_management.public.sku_unit_of_measure", connection=db_conn)
transport_asset_df = db_helper.fetch_data(sql="select * from inventory_management.public.transport_asset", connection=db_conn)


S:\git_repo\solutions-inventory-optimization\database\helper.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(


## Create necessary features, calculations




In [6]:
sku_uom_df = pd.concat(
    [
        sku_uom_df,
        sku_uom_df['pallet_dimensions'].apply(pd.Series)
    ],
    axis=1
)

sku_uom_column_mapper = {x:x for x in sku_uom_df.columns}
sku_uom_column_mapper['height_mm'] = 'pallet_height_mm'
sku_uom_column_mapper['width_mm'] = 'pallet_width_mm'
sku_uom_column_mapper['length_mm'] = 'pallet_length_mm'
sku_uom_df.rename(columns=sku_uom_column_mapper, inplace=True)

In [7]:

def create_pallet_features(
    shipment_demand_df: pd.DataFrame,
    sku_pallet_df: pd.DataFrame,           # columns from load_equipment_metadata or pallet_master
) -> pd.DataFrame:
    """
    Merges pallet/UOM master into shipment demand and computes:
      - required_pallets, full_pallets, remaining_units, partial_fill_pct
    Returns enriched shipment_candidate_df.
    """
    # Merge pallet dimensions onto demand rows
    candidate = shipment_demand_df.copy()

    # Columns expected from pallet_master: sku_id, unit_count_in_pallet,
    # pallet_height_mm, pallet_width_mm, pallet_length_mm,
    # pallet_weight_in_kg, item_weight_in_kg
    if "unit_count_in_pallet" not in candidate.columns:
        candidate = candidate.merge(
            sku_pallet_df[[
                "sku_id", "unit_count_in_pallet",
                "pallet_height_mm", "pallet_width_mm", "pallet_length_mm",
                "pallet_weight_in_kg", "item_weight_in_kg",
            ]],
            on="sku_id",
            how="left",
        )
 

    ### Filter for the ones that are divisible
    candidate = (
        candidate
        [
            candidate[
                "unit_count_in_pallet"
            ]
            > 0
        ]
    )
 
    # Compute pallet breakdown
    candidate["required_pallets"] = (
        candidate["planned_quantity"] / candidate["unit_count_in_pallet"]
    ).apply(math.ceil)
 
    candidate["full_pallets"] = (
        candidate["planned_quantity"] // candidate["unit_count_in_pallet"]
    ).astype(int)
 
    candidate["remaining_units"] = (
        candidate["planned_quantity"] % candidate["unit_count_in_pallet"]
    ).astype(int)
 
    candidate["partial_fill_pct"] = candidate.apply(
        lambda r: (r["remaining_units"] / r["unit_count_in_pallet"])
        if r["remaining_units"] > 0 else 0.0,
        axis=1,
    )
 
    return candidate
 


In [8]:
# shipment_candidate_df = (
#     create_pallet_features(
#         shipment_demand_df=shipment_demand_df,
#         sku_uom_df=sku_uom_df[['sku_id', 'unit_count_in_pallet', 'pallet_height_mm', 'pallet_width_mm', 'pallet_length_mm', 'pallet_weight_in_kg', 'item_weight_in_kg']],
#         delivery_date='2026-04-12'
#     )
# )

print('shipment_candidate_df', shipment_candidate_df.columns.to_list())

# Optimizer Flow V3

- STEP 1:
feature engineer

- STEP 2:
Placement engine

- STEP 3:
Container Fleet


## Feature Engineering

In [9]:

# ── Color palette for SKUs ────────────────────────────────────────────────────
_SKU_COLORS = [
    "#4CAF50", "#2196F3", "#FF9800", "#E91E63", "#9C27B0",
    "#00BCD4", "#FF5722", "#607D8B", "#795548", "#FFC107",
    "#3F51B5", "#8BC34A", "#F44336", "#009688", "#FFEB3B",
]



def _sku_color(sku_id: str) -> str:
    idx = int(hashlib.md5(sku_id.encode()).hexdigest(), 16) % len(_SKU_COLORS)
    return _SKU_COLORS[idx]


In [10]:

# ── Action 2: Break enriched demand rows into individual Pallet objects ───────
 
def breakdown_into_pallets(shipment_candidate_df: pd.DataFrame) -> List[Pallet]:
    """
    Explodes each demand row into N Pallet domain objects
    (full pallets + optional partial pallet).
    """
    pallets: List[Pallet] = []
    pallet_counter: Dict[str, int] = {}
 
    service_level_map = {"GOLD": 3, "SILVER": 2, "BRONZE": 1, "STANDARD": 1}
 
    for _, row in shipment_candidate_df.iterrows():
        base_id = f"{row['shipment_id']}_{row['sku_id']}"
        pallet_counter.setdefault(base_id, 0)
 
        dims = Dimension(
            depth=int(row["pallet_length_mm"]),
            width=int(row["pallet_width_mm"]),
            height=int(row["pallet_height_mm"]),
        )
        weight_per_pallet = (
            row["pallet_weight_in_kg"]
            + row["unit_count_in_pallet"] * row["item_weight_in_kg"]
        )
        service_level_val = service_level_map.get(
            str(row.get("service_level", "STANDARD")).upper(), 1
        )
        est_date = (
            pd.to_datetime(row["estimated_delivery_date"])
            if "estimated_delivery_date" in row else datetime.utcnow()
        )
 
        color = _sku_color(str(row["sku_id"]))
        temperature_req = str(row.get("temperature_requirement", "ambient") or "ambient").lower()
        special = str(row.get("special_handling", "") or "").lower()
 
        def _make_pallet(seq: int, is_partial: bool, fill_pct: float, units: int) -> Pallet:
            pallet_counter[base_id] += 1
            pid = f"PLT_{row['shipment_id']}_{row['sku_id']}_{pallet_counter[base_id]:04d}"
            w = weight_per_pallet * fill_pct if is_partial else weight_per_pallet
            return Pallet(
                candidatePalletId=pid,
                shipmentId=str(row["shipment_id"]),
                skuId=str(row["sku_id"]),
                originLocationId=str(row["origin_location_id"]),
                destinationLocationId=str(row["destination_location_id"]),
                estimatedDeliveryDate=est_date,
                dimensions=dims,
                label=str(row.get("sku_id", pid)),
                color=color,
                weightIn_kg=round(w, 2),
                priority=int(row.get("priority", 0)),
                serviceLevel=service_level_val,
                unitsInPallet=units,
                isPartialPallet=is_partial,
                fillPct=fill_pct,
                unloadSequence=int(row.get("unload_sequence_preference", 0)),
                temperatureRequirement=temperature_req,
                isFragile="fragile" in special,
                isHazmat="hazmat" in special,
            )
 
        # Full pallets
        for i in range(int(row.get("full_pallets", 0))):
            pallets.append(_make_pallet(i, False, 1.0, int(row["unit_count_in_pallet"])))
 
        # Partial pallet
        if int(row.get("remaining_units", 0)) > 0:
            fill = row["partial_fill_pct"] if row["partial_fill_pct"] > 0 else (
                row["remaining_units"] / row["unit_count_in_pallet"]
            )
            pallets.append(_make_pallet(
                int(row.get("full_pallets", 0)),
                True,
                round(fill, 4),
                int(row["remaining_units"]),
            ))
 
    return pallets
 

In [11]:

# ── Action 3: Group pallets by shipment lane ──────────────────────────────────
 
def group_pallets_by_lane(
    pallets: List[Pallet],
    lane_master_df: pd.DataFrame | None = None,
    date_granularity: str = "day",   # "day" | "week"
) -> List[ShipmentGroup]:
    """
    Groups pallets by (origin, destination, delivery_date_window).
    Optionally enriches with lane metadata.
    """
    from collections import defaultdict
 
    bucket: Dict[Tuple, List[Pallet]] = defaultdict(list)
 
    for p in pallets:
        if date_granularity == "week":
            dt_key = p.estimatedDeliveryDate.strftime("%Y-W%W")
        else:
            dt_key = p.estimatedDeliveryDate.strftime("%Y-%m-%d")
 
        key = (p.originLocationId, p.destinationLocationId, dt_key)
        bucket[key].append(p)
 
    groups: List[ShipmentGroup] = []
    for idx, ((origin, dest, dt_key), pallet_list) in enumerate(bucket.items()):
        group_id = f"GRP_{origin}_{dest}_{dt_key}".replace(" ", "_").replace("-", "")
        groups.append(ShipmentGroup(
            groupId=group_id,
            originLocationId=origin,
            destinationLocationId=dest,
            deliveryDateWindow=dt_key,
            pallets=pallet_list,
            estimatedDeliveryDate=pallet_list[0].estimatedDeliveryDate,
        ))
 
    return groups
 
 

In [12]:

# ── Action 4: Sort pallets for optimal loading sequence ───────────────────────
 
def sort_pallets_for_loading(pallets: List[Pallet], lifo: bool = True) -> List[Pallet]:
    """
    Sort order:
      1. destinationStop ASC  (multi-drop: last destination loaded first for LIFO)
      2. priority DESC
      3. serviceLevel DESC
      4. estimatedDeliveryDate ASC
      5. full pallet before partial (isPartialPallet ASC)
      6. heavier pallets first (better CoG)
 
    When LIFO is enabled, pallets for the last stop are loaded first
    (they will be unloaded last = deepest in container).
    """
    # Determine stop ordering: reverse for LIFO
    dest_order_sign = -1 if lifo else 1
 
    return sorted(
        pallets,
        key=lambda p: (
            dest_order_sign * p.destinationStop,   # LIFO: last stop loads first
            -p.priority,
            -p.serviceLevel,
            p.estimatedDeliveryDate,
            int(p.isPartialPallet),
            -p.weightIn_kg,
        ),
    )
 

## Placement Engine


Placement Engine
================
Advanced 3D bin-packing for pallets into containers.
 
Algorithm: Skyline / Column-Strip with extreme-point extension.
  - Tracks a "skyline" of occupied depth per width-strip column.
  - For each pallet: tries all valid orientations, scores positions
    by (contact surface, CoG improvement, axle balance), picks best.
  - Stacking supported optionally.
 
Actions:
  - validate_pallet_fits(container, pallet) -> (bool, str)
  - find_best_position(container, pallet) -> Position | None
  - place_pallet(container, pallet) -> bool
  - compute_axle_loads(container) -> List[AxleLoadResult]
  - compute_utilization(container, pending) -> (UtilizationMetrics, RemainingCapacity)

In [13]:

# ── Orientation helpers ───────────────────────────────────────────────────────
 
def _orientations(dim: Dimension) -> List[Tuple[int, int, int, str]]:
    """
    Returns (depth, width, height, orientation_label) for each valid rotation.
    We only allow 90° rotations around vertical axis (pallets stay upright).
    """
    d, w, h = dim.depth, dim.width, dim.height
    return [
        (d, w, h, "FRONT_FACING"),
        (w, d, h, "SIDE_FACING_LEFT"),
    ]
 


In [14]:

# ── Skyline tracker ───────────────────────────────────────────────────────────
 
class _Skyline:
    """
    Tracks occupied depth per width strip.
    Strip width = column resolution (e.g. 50 mm).
    """
    def __init__(self, container_width: int, container_depth: int, resolution: int = 50):
        self.resolution = resolution
        self.n_cols = math.ceil(container_width / resolution)
        self.container_depth = container_depth
        # skyline[col] = max depth used in that strip (from door = 0)
        self.skyline: List[int] = [0] * self.n_cols
        # 3D stack tracking: (x, z, depth, width, height) blocks
        self.blocks: List[Tuple[int, int, int, int, int, int]] = []  # x,y,z,d,w,h
 
    def _col_range(self, x_mm: int, w_mm: int) -> Tuple[int, int]:
        c0 = x_mm // self.resolution
        c1 = math.ceil((x_mm + w_mm) / self.resolution)
        return max(0, c0), min(self.n_cols, c1)
 
    def max_depth_in_strip(self, x_mm: int, w_mm: int) -> int:
        c0, c1 = self._col_range(x_mm, w_mm)
        return max(self.skyline[c0:c1]) if c0 < c1 else 0
 
    def mark_occupied(self, x_mm: int, z_mm: int, d_mm: int, w_mm: int, h_mm: int) -> None:
        c0, c1 = self._col_range(x_mm, w_mm)
        new_depth = z_mm + d_mm
        for c in range(c0, c1):
            self.skyline[c] = max(self.skyline[c], new_depth)
        self.blocks.append((x_mm, 0, z_mm, d_mm, w_mm, h_mm))
 
    def candidate_positions(self, pallet_depth: int, pallet_width: int) -> List[Tuple[int, int]]:
        """
        Returns (x, z) candidate positions using extreme-point strategy.
        All positions are integer mm, snapped to resolution grid on x-axis.
        """
        positions: set[Tuple[int, int]] = set()
        # Door-corner origin
        positions.add((0, 0))
        # Extreme points from every placed block
        for (bx, by, bz, bd, bw, bh, *_) in self.blocks:
            positions.add((bx, bz + bd))      # directly in front of block
            positions.add((bx + bw, bz))      # directly to the right of block
            positions.add((bx + bw, bz + bd)) # diagonal corner
        # Skyline step transitions
        for c in range(self.n_cols - 1):
            if self.skyline[c] != self.skyline[c + 1]:
                x_snap = c * self.resolution
                positions.add((x_snap, self.skyline[c]))
                positions.add((x_snap, self.skyline[c + 1]))
 
        cw = self.n_cols * self.resolution
        valid = []
        for (x, z) in sorted(positions):  # sort for determinism (z asc, x asc)
            x, z = int(x), int(z)
            if x < 0 or z < 0:
                continue
            if x + pallet_width > cw:
                continue
            if z + pallet_depth > self.container_depth:
                continue
            # z must be at or above the skyline for every column the pallet spans
            c0, c1 = self._col_range(x, pallet_width)
            min_z_needed = max(self.skyline[c0:c1]) if c0 < c1 else 0
            if z < min_z_needed:
                continue
            if not self._overlaps(x, z, pallet_depth, pallet_width):
                valid.append((x, z))
        return valid
 
    def _overlaps(self, x: int, z: int, d: int, w: int) -> bool:
        """Strict AABB overlap — 1mm tolerance to allow flush neighbours."""
        TOL = 1
        for (bx, by, bz, bd, bw, bh, *_) in self.blocks:
            if (x + TOL < bx + bw and x + w - TOL > bx and
                    z + TOL < bz + bd and z + d - TOL > bz):
                return True
        return False
 

In [15]:

# ── Main validation ───────────────────────────────────────────────────────────
 
def validate_pallet_fits(container: Container, pallet: Pallet) -> Tuple[bool, str]:
    """Check weight, volume, floor area, temperature, and hazmat constraints."""
    # Weight
    if container.usedWeightIn_kg + pallet.weightIn_kg > container.maxPayloadWeightIn_kg:
        return False, (
            f"Weight limit exceeded: "
            f"{container.usedWeightIn_kg + pallet.weightIn_kg:.1f} > "
            f"{container.maxPayloadWeightIn_kg:.1f} kg"
        )
    # Volume
    if container.usedVolume_m3 + pallet.volume_m3 > container.maxVolume_m3:
        return False, (
            f"Volume limit exceeded: "
            f"{container.usedVolume_m3 + pallet.volume_m3:.3f} > "
            f"{container.maxVolume_m3:.3f} m³"
        )
    # Floor area
    if container.usedFloorArea_m2 + pallet.floorArea_m2 > container.maxFloorArea_m2:
        return False, (
            f"Floor area limit exceeded: "
            f"{container.usedFloorArea_m2 + pallet.floorArea_m2:.2f} > "
            f"{container.maxFloorArea_m2:.2f} m²"
        )
    # Temperature
    if pallet.temperatureRequirement not in ("ambient", ""):
        if not container.refrigerationCapable:
            return False, "Refrigeration required but container not capable"
    # Hazmat
    if pallet.isHazmat and container.loadingRules.hazmatSegregation:
        has_non_hazmat = any(not p.isHazmat for p in container.pallets)
        if has_non_hazmat:
            return False, "Hazmat segregation violation"
 
    return True, ""

In [16]:

# ── Scoring ───────────────────────────────────────────────────────────────────
 
def _score_position(
    x: int, z: int, d: int, w: int, h: int,
    container: Container,
    skyline: _Skyline,
    pallet_weight: float,
) -> float:
    """
    Score a placement candidate. Higher = better.
    Factors:
      - Compactness: favour positions closer to door (lower z)
      - Contact surface: favour touching walls / other pallets
      - CoG balance: favour centred x
      - Axle balance: favour positions that help front/rear balance
    """
    score = 0.0
 
    # Compactness (prefer loading from the back = high z for LIFO)
    score -= z * 0.001  # slight preference for deeper placement
 
    # Centred width (better stability)
    centre_x = container.internalWidth / 2
    deviation = abs((x + w / 2) - centre_x) / container.internalWidth
    score -= deviation * 5
 
    # Wall / neighbour contact bonus
    if x == 0 or x + w >= container.internalWidth:
        score += 2
    if z == 0:
        score += 1
 
    # Axle weight balance: prefer centred depth
    depth_ratio = (z + d / 2) / container.internalDepth
    score -= abs(depth_ratio - 0.5) * 3
 
    return score
 

In [17]:

def _distribute_weight_to_axles(
    container: Container, pallet: Pallet, z_mm: int, depth_mm: int
) -> None:
    """
    Distribute pallet weight across axles based on longitudinal position.
    Simple beam formula: load on axle i proportional to proximity.
    """
    pallet_cog_z = z_mm + depth_mm / 2  # CoG from door
    total_len = container.internalDepth
 
    for axle in container.axles:
        # Proximity factor: 1 when directly over axle, 0 at the other end
        dist = abs(pallet_cog_z - axle.positionX)
        factor = max(0.0, 1 - dist / total_len)
        axle.currentLoad += pallet.weightIn_kg * factor
 
 
# ── Axle load analysis ────────────────────────────────────────────────────────
 
def compute_axle_loads(container: Container) -> List[AxleLoadResult]:
    results = []
    for axle in container.axles:
        util = (axle.currentLoad / axle.maxWeight * 100) if axle.maxWeight > 0 else 0
        results.append(AxleLoadResult(
            axleId=axle.axleId,
            currentLoad_kg=round(axle.currentLoad, 2),
            maxLoad_kg=axle.maxWeight,
            utilization_pct=round(util, 2),
            isOverloaded=axle.currentLoad > axle.maxWeight,
        ))
    return results
 

In [18]:

# ── Position finder ───────────────────────────────────────────────────────────
 
def find_best_position(
    container: Container,
    pallet: Pallet,
    skyline: _Skyline,
) -> Optional[Position]:
    """
    Try all orientations × candidate positions, return best Position or None.
    """
    iW = int(container.internalWidth)
    iD = int(container.internalDepth)
    iH = int(container.internalHeight)
 
    best_pos: Optional[Position] = None
    best_score = float("-inf")
 
    for (dep, wid, hgt, orient) in _orientations(pallet.dimensions):
        # Height check
        if hgt > iH:
            continue
        # Door width / height check for first pallet
        if dep > container.doorHeight or wid > container.doorWidth:
            pass  # still ok if not first; door constraint mainly for forklift entry
 
        candidates = skyline.candidate_positions(dep, wid)
        if not candidates:
            # Fallback: place at current depth position
            candidates = [(0, skyline.max_depth_in_strip(0, iW))]
 
        for (x, z) in candidates:
            # Boundary checks
            if x + wid > iW or z + dep > iD:
                continue
            # Overlap check already done in skyline, double-check height
            score = _score_position(x, z, dep, wid, hgt, container, skyline, pallet.weightIn_kg)
            if score > best_score:
                best_score = score
                best_pos = Position(
                    x=x, y=0, z=z,
                    orientation=orient,
                    effectiveWidth=wid,   # actual x-footprint after rotation
                    effectiveDepth=dep,   # actual z-footprint after rotation
                    effectiveHeight=hgt,  # height unchanged
                )
                # Also stash on __dict__ for use in place_pallet mark_occupied call
                best_pos.__dict__["_dep"] = dep
                best_pos.__dict__["_wid"] = wid
                best_pos.__dict__["_hgt"] = hgt
 
    return best_pos
 

In [19]:

# ── Place pallet action ───────────────────────────────────────────────────────
 
# Module-level skylines keyed by container id
_skylines: dict[str, _Skyline] = {}
 
 
def _get_skyline(container: Container) -> _Skyline:
    if container.containerId not in _skylines:
        _skylines[container.containerId] = _Skyline(
            int(container.internalWidth), int(container.internalDepth)
        )
    return _skylines[container.containerId]
 
 
def reset_skyline(container_id: str) -> None:
    _skylines.pop(container_id, None)
 
 
def place_pallet(container: Container, pallet: Pallet) -> bool:
    """
    Attempt to place pallet into container.
    Mutates container state and pallet.position on success.
    Returns True if placed, False otherwise.
    """
    ok, reason = validate_pallet_fits(container, pallet)
    if not ok:
        pallet.rejectionReason = reason
        return False
 
    skyline = _get_skyline(container)
    pos = find_best_position(container, pallet, skyline)
 
    if pos is None:
        pallet.rejectionReason = "No valid position found in container"
        return False
 
    # Extract chosen dims
    dep = pos.__dict__.get("_dep", pallet.dimensions.depth)
    wid = pos.__dict__.get("_wid", pallet.dimensions.width)
    hgt = pos.__dict__.get("_hgt", pallet.dimensions.height)
 
    # Mark position
    pallet.position = pos
    pallet.loadedToContainer = True
 
    # Update container state
    container.usedWeightIn_kg += pallet.weightIn_kg
    container.usedFloorArea_m2 += pallet.floorArea_m2
    container.usedVolume_m3 += pallet.volume_m3
    container.loadedPallets += 1
 
    # Mark skyline
    skyline.mark_occupied(pos.x, pos.z, dep, wid, hgt)
 
    # Update axle loads
    _distribute_weight_to_axles(container, pallet, pos.z, dep)
 
    return True
 

In [20]:

# ── Utilization ───────────────────────────────────────────────────────────────
 
def compute_utilization(
    container: Container, pending: List[Pallet]
) -> Tuple[UtilizationMetrics, RemainingCapacity]:
    wt_util = (container.usedWeightIn_kg / container.maxPayloadWeightIn_kg * 100
               if container.maxPayloadWeightIn_kg else 0)
    vol_util = (container.usedVolume_m3 / container.maxVolume_m3 * 100
                if container.maxVolume_m3 else 0)
    area_util = (container.usedFloorArea_m2 / container.maxFloorArea_m2 * 100
                 if container.maxFloorArea_m2 else 0)
 
    util = UtilizationMetrics(
        weightUtilization_pct=round(wt_util, 2),
        volumeUtilization_pct=round(vol_util, 2),
        floorAreaUtilization_pct=round(area_util, 2),
        loadedPallets=container.loadedPallets,
        totalPallets=container.loadedPallets + len(pending),
        usedWeightIn_kg=round(container.usedWeightIn_kg, 2),
        usedVolumeIn_m3=round(container.usedVolume_m3, 4),
        usedFloorAreaIn_m2=round(container.usedFloorArea_m2, 4),
        maxWeightIn_kg=container.maxPayloadWeightIn_kg,
        maxVolumeIn_m3=round(container.maxVolume_m3, 4),
        maxFloorAreaIn_m2=round(container.maxFloorArea_m2, 4),
    )
    remaining = RemainingCapacity(
        remainingWeight_kg=round(container.maxPayloadWeightIn_kg - container.usedWeightIn_kg, 2),
        remainingVolume_m3=round(container.maxVolume_m3 - container.usedVolume_m3, 4),
        remainingFloorArea_m2=round(container.maxFloorArea_m2 - container.usedFloorArea_m2, 4),
    )
    return util, remaining

## Container Fleet Optimizer

Orchestrates the full load planning pipeline:
Actions (function calls):
 - load_equipment_to_container_spec(row)
 - open_new_container(spec, container_idx, group)
 - load_pallets_into_container(container, pallets) -> ContainerOptimizationResult
 - optimize_container_fleet(group, equipment_df, rules) -> ContainerFleetOptimizationResult
 - run_full_optimization(shipment_candidate_df, ...) -> ContainerFleetOptimizationResult
 - export_container_json(result, out_dir) -> str

In [21]:

 
# ── Action: Map equipment row → Container spec ────────────────────────────────
 
def load_equipment_to_container_spec(row: pd.Series) -> Dict[str, Any]:
    """Convert a load_equipment_metadata_df row to Container constructor kwargs."""
    axle_conf = str(row.get("axle_configuration", "single")).lower()
    axles = []
    if "tandem" in axle_conf or "2" in axle_conf:
        axles = [
            Axle(axleId="FRONT", maxWeight=10000, positionX=float(row.get("internal_length_mm", 12000)) * 0.15),
            Axle(axleId="REAR", maxWeight=20000, positionX=float(row.get("internal_length_mm", 12000)) * 0.75),
        ]
    else:
        axles = [Axle(axleId="DEFAULT", maxWeight=30000,
                       positionX=float(row.get("internal_length_mm", 12000)) * 0.45)]
 
    return dict(
        containerType=str(row.get("equipment_name", "CONTAINER")),
        containerDepth=float(row.get("length_mm", 12191)),
        containerWidth=float(row.get("width_mm", 2438)),
        containerHeight=float(row.get("height_mm", 2591)),
        internalDepth=float(row.get("internal_length_mm", 11836)),
        internalWidth=float(row.get("internal_width_mm", 2352)),
        internalHeight=float(row.get("internal_height_mm", 2391)),
        maxPayloadWeightIn_kg=float(row.get("max_payload_weight_kg", 25000)),
        tareWeightIn_kg=float(row.get("tare_weight_kg", 2000)),
        doorWidth=float(row.get("door_width_mm", 2352)),
        doorHeight=float(row.get("door_height_mm", 2391)),
        refrigerationCapable=bool(row.get("refrigeration_capable", False)),
        temperatureMin_c=row.get("temperature_min_c"),
        temperatureMax_c=row.get("temperature_max_c"),
        axles=axles,
        loadingRules=ContainerLoadingRules(
            maxStackHeightIn_mm=float(row.get("max_stack_height_mm", 0)) or 0,
            allowStacking=float(row.get("max_stack_height_mm", 0) or 0) > 0,
        ),
    )
 

In [22]:

# ── Action: Open a fresh container for a group ────────────────────────────────
 
def open_new_container(
    spec: Dict[str, Any],
    container_idx: int,
    group: ShipmentGroup,
) -> Container:
    """Instantiate a new Container with summary pre-populated from the lane group."""
    cid = f"CONT_{group.originLocationId}_{group.destinationLocationId}_{container_idx:03d}"
    summary = ContainerSummary(
        routeId=group.groupId,
        origin=group.originLocationId,
        destinationInSequence=[group.destinationLocationId],
    )
    container = Container(
        containerId=cid,
        pallets=[],
        summary=summary,
        **spec,
    )
    return container
 

In [23]:

# ── Action: Load sorted pallets into a single container ───────────────────────
 
def load_pallets_into_container(
    container: Container,
    pallets: List[Pallet],
) -> ContainerOptimizationResult:
    """
    Greedily loads pallets into container until no more fit.
    Returns a ContainerOptimizationResult with loaded / pending pallets.
    """
    reset_skyline(container.containerId)
    loaded: List[Pallet] = []
    pending: List[Pallet] = []
 
    for pallet in pallets:
        if place_pallet(container, pallet):
            container.pallets.append(pallet)
            loaded.append(pallet)
        else:
            pending.append(pallet)
 
    # Update container summary
    container.summary.totalPallets = len(loaded)
    container.summary.totalWeightIn_kg = round(container.usedWeightIn_kg, 2)
    container.summary.totalVolumeIn_m3 = round(container.usedVolume_m3, 4)
 
    axle_loads = compute_axle_loads(container)
    utilization, remaining = compute_utilization(container, pending)
 
    return ContainerOptimizationResult(
        container=container,
        loadedPallets=loaded,
        pendingPallets=pending,
        axleLoads=axle_loads,
        utilization=utilization,
        remainingCapacity=remaining,
    )
 
 

In [24]:

# ── Action: Optimize full fleet for a shipment group ─────────────────────────
 
def optimize_container_fleet(
    group: ShipmentGroup,
    equipment_spec: Dict[str, Any],
    lifo: bool = True,
    max_containers: int = 10,
) -> ContainerFleetOptimizationResult:
    """
    Creates as many containers as needed to load all pallets in the group.
    Current version: infinite containers available.
    """
    sorted_pallets = sort_pallets_for_loading(group.pallets, lifo=lifo)
    remaining = list(sorted_pallets)
 
    all_containers: List[Container] = []
    all_results: List[ContainerOptimizationResult] = []
    container_idx = 1
 
    while remaining and container_idx <= max_containers:
        container = open_new_container(equipment_spec, container_idx, group)
        result = load_pallets_into_container(container, remaining)
 
        all_containers.append(container)
        all_results.append(result)
 
        # Pallets still pending after this container
        remaining = result.pendingPallets
 
        if not result.loadedPallets:
            # No pallet fit at all — break to avoid infinite loop
            break
 
        container_idx += 1
 
    # Fleet metrics
    total_wt = sum(c.usedWeightIn_kg for c in all_containers)
    total_max_wt = sum(c.maxPayloadWeightIn_kg for c in all_containers)
    total_vol = sum(c.usedVolume_m3 for c in all_containers)
    total_max_vol = sum(c.maxVolume_m3 for c in all_containers)
 
    return ContainerFleetOptimizationResult(
        containers=all_containers,
        containerResults=all_results,
        unallocated_pallets=remaining,
        total_pallets=len(group.pallets),
        total_containers=len(all_containers),
        fleet_weight_utilization_pct=round(total_wt / total_max_wt * 100, 2) if total_max_wt else 0,
        fleet_volume_utilization_pct=round(total_vol / total_max_vol * 100, 2) if total_max_vol else 0,
        optimizer_run_id=uuid.uuid4().hex,
    )
 



In [25]:

# ── Action: Full pipeline entry point ─────────────────────────────────────────
 
def run_full_optimization(
    shipment_demand_df: pd.DataFrame,
    sku_pallet_df: pd.DataFrame,
    load_equipment_metadata_df: pd.DataFrame,
    lane_master_df: Optional[pd.DataFrame] = None,
    preferred_equipment_type: str = "CONTAINER",
    lifo: bool = True,
) -> Dict[str, ContainerFleetOptimizationResult]:
    """
    End-to-end optimization:
      1. Feature engineering → pallets
      2. Group by lane
      3. Per group: fleet optimization
    Returns dict keyed by groupId.
    """
    # Step 1 — Enrich demand with pallet features
    candidate_df = create_pallet_features(shipment_demand_df, sku_pallet_df)
 
    # Step 2 — Breakdown into Pallet objects
    all_pallets = breakdown_into_pallets(candidate_df)
 
    # Step 3 — Group by lane
    groups = group_pallets_by_lane(all_pallets, lane_master_df)
 
    # Step 4 — Select default equipment spec
    eq_row = load_equipment_metadata_df[
        load_equipment_metadata_df["equipment_type"].str.upper() == preferred_equipment_type.upper()
    ].iloc[0] if len(load_equipment_metadata_df) else pd.Series()
 
    if eq_row.empty:
        eq_row = load_equipment_metadata_df.iloc[0]
 
    equipment_spec = load_equipment_to_container_spec(eq_row)
 
    # Step 5 — Optimize each group
    results: Dict[str, ContainerFleetOptimizationResult] = {}
    for group in groups:
        fleet_result = optimize_container_fleet(
            group, equipment_spec, lifo=lifo
        )
        results[group.groupId] = fleet_result
 
    return results
 

In [26]:

# ── Action: Export to JSON ────────────────────────────────────────────────────
 
def _pallet_to_viz_dict(p: Pallet) -> dict:
    return {
        "dimensions": {
            "depth": p.dimensions.depth,
            "width": p.dimensions.width,
            "height": p.dimensions.height,
        },
        "position": {
            "x": p.position.x,
            "y": p.position.y,
            "z": p.position.z,
        },
        "label": p.label or p.skuId,
        "color": p.color,
        "weightIn_kg": p.weightIn_kg,
        "isPartialPallet": p.isPartialPallet,
        "fillPct": p.fillPct,
        "shipmentId": p.shipmentId,
        "skuId": p.skuId,
        "priority": p.priority,
        "destinationStop": p.destinationStop,
        "unloadSequence": p.unloadSequence,
    }
 

In [27]:

from utils.CustomJSONEncoder import CustomJSONEncoder


def export_container_json(
    fleet_result: ContainerFleetOptimizationResult,
    out_dir: str = "./output",
    group_id: str = "default",
) -> List[str]:
    """
    Dumps each container as a visualization-ready JSON file.
    Returns list of file paths written.
    """
    os.makedirs(out_dir, exist_ok=True)
    paths = []
    
    for cr in fleet_result.containerResults:
        c = cr.container

        fname = f"{group_id}_{c.containerId}.json"
        fpath = os.path.join(out_dir, fname)
        with open(fpath, "w") as f:
            json.dump(c, f, indent=2, cls=CustomJSONEncoder)
        paths.append(fpath)

    return paths
 

In [28]:
result = run_full_optimization(
    shipment_demand_df=shipment_demand_df,
    sku_pallet_df=sku_uom_df,
    load_equipment_metadata_df=load_equipment_metadata_df,
    lane_master_df=lane_master_df,
    preferred_equipment_type='CONTAINER',
    lifo=True,
)

In [29]:
type(result['GRP_151_0152_20260410'])

objects.ContainerFleetOptimizationResult.ContainerFleetOptimizationResult

In [30]:
result['GRP_151_0152_20260410'].containerResults[0].container

Container(containerId='CONT_151_0152_001', containerType='CONTAINER 40FT GP', containerDepth=12192.0, containerWidth=2438.0, containerHeight=2591.0, internalDepth=12031.0, internalWidth=2352.0, internalHeight=2393.0, maxPayloadWeightIn_kg=25000.0, tareWeightIn_kg=0.0, currentVolume_m3=0, unit='mm', doorWidth=2340.0, doorHeight=2280.0, axles=[Axle(axleId='DEFAULT', maxWeight=30000.0, positionX=5413.95, currentLoad=148737220.97326526, utilizationPct=0)], pallets=[Pallet(candidatePalletId='PLT_Lane_From_0151_To_0152-0001-207410-10-Apr-26-1_207410_0001', shipmentId='Lane_From_0151_To_0152-0001-207410-10-Apr-26-1', skuId='207410', originLocationId='151', destinationLocationId='0152', estimatedDeliveryDate=Timestamp('2026-04-10 00:00:00'), dimensions=Dimension(depth=1219, width=1016, height=150), position=Position(x=0, y=0, z=0, orientation='SIDE_FACING_LEFT', effectiveWidth=1219, effectiveDepth=1016, effectiveHeight=150), label='207410', color='#FF9800', weightIn_kg=3.34, floorArea_m2=1.238

In [33]:
for k in result.keys():
    export_container_json(result[k])

In [32]:
break

SyntaxError: 'break' outside loop (668683560.py, line 1)

# Optimizer Flow V2

- STEP 1:
build_shipment_candidates()

- STEP 2:
expand_pallet_candidates()

- STEP 3:
initialize_container()

- STEP 4:
build_container_load()

```
    while capacity exists:
        find_next_position()
        calculate_axle_loads()
        validate_constraints()
        load pallet
```
- STEP 5
build_pending_loads()

- STEP 6
build_container_objects()

- STEP 7
visualize()

In [ ]:
def build_fixed_container(
	load_equipment_metadata_df, container_name: str='40FT'
):
	"""
	Build fixed 40FT container.
	
	Returns
	-------
	Container
	"""

	equipment_row = (
		load_equipment_metadata_df
		.loc[
			load_equipment_metadata_df[
				"equipment_name"
			]
			.str.upper()
			.str.contains(
				container_name,
				na=False
			)
		]
		.iloc[0]
	)

	container = Container(

		containerId=str(
			equipment_row["equipment_id"]
		),

		containerType=
			equipment_row["equipment_name"],

		depth=
			equipment_row["length_mm"],

		width=
			equipment_row["width_mm"],

		height=
			equipment_row["height_mm"],

		internalDepth=
			equipment_row["internal_length_mm"],

		internalWidth=
			equipment_row["internal_width_mm"],

		internalHeight=
			equipment_row["internal_height_mm"],

		maxPayloadWeightIn_kg=
			equipment_row["max_payload_weight_kg"],

		tareWeightIn_kg=
			equipment_row["tare_weight_kg"],

		maxVolume_m3=(
			equipment_row["internal_length_mm"]
			*
			equipment_row["internal_width_mm"]
			*
			equipment_row["internal_height_mm"]
		) / 1_000_000_000,

		doorWidth=
			equipment_row["door_width_mm"],

		doorHeight=
			equipment_row["door_height_mm"],

		pallets=[],
		axles=[{
				'axleId':"FRONT_ZONE",
				'maxWeightIn_kg':16000,
				'positionXFromFrontIn_mm':3000
			},{
				"axleId":"REAR_ZONE",
				"maxWeightIn_kg":24000,
				"positionXFromFrontIn_mm":10000
			}
		]
	)

	return container


In [ ]:


container_40ft = build_fixed_container(load_equipment_metadata_df=load_equipment_metadata_df)

In [ ]:

shipment_candidate_df[
    [
        "sku_id",
        "planned_quantity",
        "unit_count_in_pallet",
        "required_pallets",
        "full_pallets",
        "remaining_units",
        "partial_fill_pct"
    ]
].head()

In [ ]:

import uuid


def expand_pallet_candidates(
        shipment_candidate_df,
        partial_pallet_threshold=0.80
):

    pallet_candidates = []

    for _, row in shipment_candidate_df.iterrows():

        # --------------------------
        # Full Pallets
        # --------------------------

        for _ in range(
                int(row["full_pallets"])
        ):

            unitsInPallet=int(row["unit_count_in_pallet"])
            isPartialPallet=False
            weightIn_kg = float(row["pallet_weight_in_kg"])
            fillPct = 1.0
            rejectionReason = ""

            _pallet_candidate_: Pallet = Pallet(
                candidatePalletId=f"PALLET_{uuid.uuid4().hex[:12].upper()}",
                shipmentId=str(row["shipment_id"]),
                skuId=str(row["sku_id"]),
                originLocationId=str(row["origin_location_id"]),
                destinationLocationId=str(row["destination_location_id"]),
                estimatedDeliveryDate=row["estimated_delivery_date"],
                priority=int(row["priority"]),
                serviceLevel=int(row["service_level"]),
                unitsInPallet=unitsInPallet, # partial pallet / full pallet
                isPartialPallet=isPartialPallet, # partial pallet / full pallet
                fillPct=fillPct, # partial pallet / full pallet
                weightIn_kg=weightIn_kg, # partial pallet / full pallet
                floorArea_m2=(
                    row["pallet_length_mm"]
                    *
                    row["pallet_width_mm"]
                ) / 1_000_000,

                volume_m3=(
                    row["pallet_length_mm"]
                    *
                    row["pallet_width_mm"]
                    *
                    row["pallet_height_mm"]
                ) / 1_000_000_000,

                dimensions={
                    'depth':int(
                        row["pallet_length_mm"]
                    ),
                    'width':int(
                        row["pallet_width_mm"]
                    ),
                    'height':int(
                        row["pallet_height_mm"]
                    )
                },

                position={
                    'x':0,
                    'y':0,
                    'z':0
                },

                label=f"SKU {row['sku_id']}",
                rejectionReason=rejectionReason,
            )
            pallet_candidates.append(_pallet_candidate_)


            # --------------------------
            # Partial Pallet
            # --------------------------

            if (
                    row["partial_fill_pct"]
                    >=
                    partial_pallet_threshold
            ):
                fillPct = float(
                    row["partial_fill_pct"]
                )
                isPartialPallet=True
                unitsInPallet=int(row["remaining_units"])
                weightIn_kg *= weightIn_kg

                _pallet_candidate_.fillPct = fillPct
                _pallet_candidate_.isPartialPallet = isPartialPallet
                _pallet_candidate_.unitsInPallet = unitsInPallet
                _pallet_candidate_.weightIn_kg = weightIn_kg
                pallet_candidates.append(_pallet_candidate_)

    return pallet_candidates


In [ ]:
pallet_candidates = (
    expand_pallet_candidates(
        shipment_candidate_df=shipment_candidate_df
    )
)

print(
    len(
        pallet_candidates
    )
)

# pallet_candidate_df.head()

In [ ]:
# Step 3A: Build Candidate Queue

from typing import List


def build_pallet_candidate_queue(
    pallet_candidates: List[Pallet]
):
    return sorted(
        pallet_candidates,
        key=lambda x: (
            -x.priority,
            -x.serviceLevel,
            x.estimatedDeliveryDate,
            x.isPartialPallet
        )
    )

In [ ]:
# Step 3C - Container Validation function


def can_fit_candidate_in_container(
        candidate: Pallet,
        container: Container,
):
    """
    Checks are related to weight, volume, floor space volume

    Args:
        candidate (Pallet): _description_
        container (Container): _description_

    Returns:
        _type_: _description_
    """

    max_floor_area = container.maxFloorArea_m2

    max_volume = container.maxVolume_m3

    return (
        (container.usedWeightIn_kg
         + candidate.weightIn_kg)
        <=
        container.maxPayloadWeightIn_kg
        and
        (
            container.usedFloorArea_m2
            + candidate.floorArea_m2
        ) <=
        max_floor_area
        and
        (container.usedVolume_m3
         + candidate.volume_m3
         ) <=
        max_volume
    )


#Step 4: Optimizer pseudo code

```
while candidates remain:

    candidate = next candidate

    position = find_next_position()

    axle_loads = calculate_axle_loads()

    if valid:

        load pallet

    else:

        skip pallet

end 
```


```
Pick Candidate
    ↓
Find Position
    ↓
Check Constraints
    ↓
Check Axle Loads
    ↓
Load
```

In [ ]:
def find_next_position(
    candidate: Pallet,
    container: Container,
):

    pallet_depth = (
        candidate.dimensions.depth
    )

    pallet_width = (
        candidate.dimensions.width
    )

    next_depth = (
        container.currentDepthPosition_mm
    )

    next_width = (
        container.currentWidthPosition_mm
    )

    # New row

    if (
        next_width
        +
        pallet_width
        >
        container.internalWidth
    ):
        next_depth += (
            container.currentRowDepth_mm
        )
        next_width = 0

    # Container full

    if (
        next_depth
        +
        pallet_depth
        >
        container.internalDepth

    ):
        return None

    return Position(
        x=int(next_depth),
        y=0,
        z=int(next_width)
    )

In [ ]:
# Step 4.2 Calculate Axle Loads

def calculate_axle_loads(
    container
):

    axle_loads = {
        axle.axleId: 0
        for axle in container.axles
    }

    if len(container.axles) < 2:
        return axle_loads

    front_axle = (
        container.axles[0]
    )

    rear_axle = (
        container.axles[1]
    )

    span = (
        rear_axle.positionXFromFrontIn_mm
        -
        front_axle.positionXFromFrontIn_mm
    )

    for pallet in container.pallets:

        pallet_cg_x = (
            pallet.position.x
            +
            pallet.dimensions.depth / 2
        )

        weight = (
            pallet.weightIn_kg
        )

        front_distance = abs(
            pallet_cg_x
            -
            front_axle.positionXFromFrontIn_mm
        )

        rear_distance = abs(
            rear_axle.positionXFromFrontIn_mm
            -
            pallet_cg_x
        )

        front_reaction = (
            weight
            *
            rear_distance
            /
            span
        )

        rear_reaction = (
            weight
            *
            front_distance
            /
            span
        )

        axle_loads[
            front_axle.axleId
        ] += front_reaction

        axle_loads[
            rear_axle.axleId
        ] += rear_reaction

    return axle_loads

In [ ]:
def load_candidate(
    candidate: Pallet,
    position: Position,
    container: Container,
):

    candidate.position = position

    container.pallets.append(
        candidate
    )

    container.usedWeightIn_kg += (
        candidate.weightIn_kg
    )

    container.usedFloorArea_m2 += (
        candidate.floorArea_m2
    )

    container.usedVolume_m3 += (
        candidate.volume_m3
    )

    container.loadedPallets += 1

    # --------------------------------
    # Detect row change
    # --------------------------------

    if (
        position.z == 0
        and
        container.currentWidthPosition_mm > 0
    ):
        container.currentDepthPosition_mm += (
            container.currentRowDepth_mm
        )
        container.currentWidthPosition_mm = 0
        container.currentRowDepth_mm = 0

    # --------------------------------
    # Advance cursor
    # --------------------------------

    container.currentWidthPosition_mm += (
        candidate.dimensions.width
    )

    container.currentRowDepth_mm = max(
        container.currentRowDepth_mm,
        candidate.dimensions.depth
    )

In [ ]:
# Step 5.1 Axle Validation

def validate_axle_limits(
    container
):

    axle_loads = (
        calculate_axle_loads(
            container
        )
    )

    for axle in container.axles:
        current_load = (
            axle_loads.get(
                axle.axleId,
                0
            )
        )

        if (
            current_load
            >
            axle.maxWeightIn_kg
        ):
            return False
    return True

In [ ]:
# Step 5.2 Try load candidate
from copy import deepcopy
def try_load_candidate(
    candidate,
    position,
    container,
):
    # Copy of entire data
    # Might cause memory issue?
    container_backup = deepcopy(
        container
    )

    load_candidate(
        candidate=candidate,
        position=position,
        container=container,
    )

    if validate_axle_limits(
        container
    ):
        return True

    container.pallets = (
        container_backup.pallets
    )

    return False

In [ ]:
# Step 5.3 Capacity remaining


def calculate_remaining_capacity(
    container: Container,
):

    return {
        "weight_kg":
            container.maxPayloadWeightIn_kg
            -
            container.usedWeightIn_kg,

        "floor_area_m2":
            container.maxFloorArea_m2
            -
            container.usedFloorArea_m2,

        "volume_m3":
            container.maxVolume_m3
            -
            container.usedVolume_m3
    }

In [ ]:
# Step 5.4 Utilization Metrics

def calculate_utilization_metrics(
    container: Container,
):

    return {
        "weight_utilization_pct":
            round(
                container.usedWeightIn_kg
                /
                container.maxPayloadWeightIn_kg
                *
                100,
                2
            ),

        "floor_utilization_pct":
            round(
                container.usedFloorArea_m2
                /
                container.maxFloorArea_m2
                *
                100,
                2
            ),

        "volume_utilization_pct":
            round(
                container.usedVolume_m3
                /
                container.maxVolume_m3
                *
                100,
                2
            )
    }


In [ ]:
from typing import List

from objects.ContainerOptimizationResult import ContainerOptimizationResult


def build_container_load(
    candidate_queue: List[Pallet],
    container: Container,
) -> ContainerOptimizationResult:
    """
    Build container using pallet candidates.

    Returns
    -------
    ContainerOptimizationResult
    """

    loaded_candidates = []

    pending_candidates = []

    for candidate in candidate_queue:

        # ------------------------------------
        # Find position
        # ------------------------------------

        position = find_next_position(
            candidate=candidate,
            container=container,999
        )

        # Container full
        if position is None:

            candidate.rejectionReason = "NO_SPACE_AVAILABLE"

            pending_candidates.append(
                candidate
            )

            continue

        # ------------------------------------
        # Capacity validation
        # ------------------------------------

        if not can_fit_candidate_in_container(
            candidate=candidate,
            container=container,
        ):

            candidate.rejectionReason = "CAPACITY_EXCEEDED"

            pending_candidates.append(
                candidate
            )

            continue

        # ------------------------------------
        # Load candidate and validate axle
        # ------------------------------------

        loaded = try_load_candidate(
            candidate=candidate,
            position=position,
            container=container,
        )

        if not loaded:

            candidate.rejectionReason = "AXLE_LIMIT_EXCEEDED"

            pending_candidates.append(
                candidate
            )

            continue

        loaded_candidates.append(
            candidate
        )

    # ----------------------------------------
    # Update container summary
    # ----------------------------------------

    container.summary.totalPallets = (
        len(container.pallets)
    )

    container.summary.totalWeightIn_kg = round(
        container.usedWeightIn_kg,
        2
    )

    container.summary.totalVolumeIn_m3 = round(
        container.usedVolume_m3,
        4
    )

    container.currentVolume_m3 = round(
        container.usedVolume_m3,
        4
    )

    # ----------------------------------------
    # Build result
    # ----------------------------------------

    result = ContainerOptimizationResult(
        container=container,
        loadedPallets=loaded_candidates,
        pendingPallets=pending_candidates,
        axleLoads=calculate_axle_loads(
            container=container
        ),
        utilization=calculate_utilization_metrics(
            container=container,
        ),
        remainingCapacity=calculate_remaining_capacity(
            container=container,
        )
    )

    return result

In [ ]:
# Step 5.5 Build Pending backlogs

def build_pending_load_df(
    pending_candidates
):

    rows = []

    for pallet in pending_candidates:

        rows.append({

            "shipment_id":
                pallet.shipmentId,

            "sku_id":
                pallet.skuId,

            "units":
                pallet.unitsInPallet,

            "weight_kg":
                pallet.weightIn_kg,

            "reason":
                "CONTAINER_CAPACITY"
        })

    return pd.DataFrame(
        rows
    )

In [ ]:
total_weight = sum(
    p.weightIn_kg
    for p in pallet_candidates
)

print(
    "Candidate Weight:",
    round(total_weight, 2)
)

In [ ]:
print(container_40ft.containerType)

print(
    container_40ft.maxPayloadWeightIn_kg
)

print(
    container_40ft.usedWeightIn_kg
)

In [ ]:
pallet_candidate_queue = (
    build_pallet_candidate_queue(
        pallet_candidates=pallet_candidates
    )
)

print(
    f"Queue Size: "
    f"{len(pallet_candidate_queue):,}"
)

In [ ]:
result = (
    build_container_load(
        candidate_queue=pallet_candidate_queue,
        container=container_40ft,
    )
)

In [ ]:

result

In [ ]:
print(result.utilization)

print(result.axleLoads)

print(len(result.loadedPallets))

print(len(result.pendingPallets))

In [ ]:
temp_res = deepcopy(result)
temp_res.container.pallets = temp_res.container.pallets[:3]

In [ ]:
import json

from utils.CustomJSONEncoder import CustomJSONEncoder

with open("Result.json", 'w') as f:
    json.dump(result, f, cls=CustomJSONEncoder, )

In [ ]:
import json

from utils.CustomJSONEncoder import CustomJSONEncoder

with open("res.json", 'w') as f:
    json.dump(result.container, f, cls=CustomJSONEncoder, )

In [ ]:
for pallet in result.container.pallets:

    print(
        pallet.label,
        pallet.position.x,
        pallet.position.y,
        pallet.position.z
    )

In [ ]:
# container_visualization(
#     temp_res.container
# )

## Container Fleet Loading Strategy

In [ ]:
from copy import deepcopy

from objects.ContainerFleetOptimizationResult import ContainerFleetOptimizationResult


def optimize_container_fleet(
    candidate_queue,
    container_template
):

    containers = []

    remaining_candidates = (
        candidate_queue.copy()
    )

    container_no = 1

    while len(remaining_candidates) > 0:

        print(
            f"Building Container "
            f"{container_no}"
        )

        container = deepcopy(
            container_template
        )

        container.containerId = (
            f"CONT_{container_no:03d}"
        )

        result = (
            build_container_load(
                candidate_queue=remaining_candidates,
                container=container,
            )
        )

        containers.append(
            result.container
        )

        loaded_ids = {
            pallet.candidatePalletId
            for pallet in
            result.loadedPallets
        }

        remaining_candidates = [
            pallet
            for pallet in
            remaining_candidates
            if pallet.candidatePalletId
            not in loaded_ids
        ]

        # if len(result.loadedPallets) == 0:

        #     print(
        #         "No progress possible."
        #     )

        #     break

        container_no += 1

    return ContainerFleetOptimizationResult(
        containers=containers,
        unallocated_pallets=remaining_candidates,
        total_pallets=len(candidate_queue),
        total_containers=len(containers)
    )

In [ ]:
all_containers_result = optimize_container_fleet(candidate_queue=pallet_candidate_queue, container_template=container_40ft)

In [ ]:
break

## Create Links for the following
1. Transport equipment assignment 🚚
   1. (transport_asset_df + load_equipment) [🛻+📦 record] 
2. 